# IDX Stock Scoring Model

## Overview
This notebook ranks Indonesia Stock Exchange (IDX) equities using a transparent multi-factor scoring framework. It combines market data and company metrics into a single **Total Score (0-100)** for screening and prioritization.

## Factor Framework and Weights
- Quality: 25%
- Growth: 25%
- Income: 20%
- Valuation: 15%
- Risk: 10%
- Momentum: 5%

Scoring formula:

`Total Score = sum(Factor Score x Weight)`

## Data Sources
- IDX universe and metadata: `pasardana.id`
- Market and fundamentals data: Yahoo Finance (`yfinance`)

## Output
The final output is a ranked stock table (`ranked_stocks`) with both aggregate and per-factor scores, plus optional sector-level ranking views.

## Notes
- This model is for research and educational use.
- Scores are data-dependent and can change as source data updates.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)


## 1) Load IDX Metrics

Data is produced by `fetch_data.py` (Pasardana universe -> liquidity pre-filter -> concurrent, cached Yahoo fundamentals). Run `.venv/bin/python fetch_data.py` to refresh `data/latest.csv`; see `data/coverage_latest.csv` for per-field coverage.


In [2]:
merged_df = pd.read_csv('data/latest.csv')
print(merged_df.shape, merged_df['Fetched On'].iloc[0])
merged_df.head()


(356, 78) 2026-10-21


,Code,Name,SectorName,SubSectorName,NewSectorName,NewIndustryName,NewSubSectorName,NewSubIndustryName,Last,PrevClosingPrice,AdjustedClosingPrice,Volume,Frequency,Value,OneDay,OneWeek,OneMonth,ThreeMonth,SixMonth,OneYear,ThreeYear,FiveYear,TenYear,Mtd,Ytd,Per,Pbr,Capitalization,BetaOneYear,StdevOneYear,PerAnnualized,PsrAnnualized,PcfrAnnualized,AdjustedAnnualHighPrice,AdjustedAnnualLowPrice,LastDate,Roe,FreeFloatPct,Ticker,YF Name,Sector,Current Price,Market Cap,PE Ratio,Forward PE,Dividend Yield,Debt to Equity,Revenue Growth,EPS Growth,ROA,Operating Margin,Net Margin,EV/EBITDA,Beta,Payout Ratio,Current Ratio,Quick Ratio,Free Float Ratio,Currency,Financial Currency,Revenue CAGR 3Y,Net Income CAGR 3Y,Net Income TTM,Net Income Years Positive,FCF,FCF Margin,CFO to Net Income,Interest Coverage,Debt to Equity Calc,Liabilities to Equity,Debt to Assets,Net Debt to EBITDA,Dividend TTM,Dividend Yield Calc,Dividend Years 5Y,Payout Ratio Calc,Statements Latest FY,Fetched On
0,AALI,Astra Agro Lestari Tbk.,AGRICULTURE,Plantation,Barang Konsumen Primer,Produk Makanan Pertanian,Makanan & Minuman,Perkebunan & Tanaman Pangan,8800.0,8275.0,8800.0,5156100.0,3240.0,4.411449e+10,0.063444,0.017341,0.053892,0.353846,0.047619,0.096573,0.243816,-0.141463,-0.397260,0.100000,0.193220,9.06472,0.68833,1.693726e+13,0.639747,0.333949,7.70654,0.55499,2.53527,8800.0,5825.0,2026-10-21T00:00:00,0.089317,20.27,AALI.JK,Astra Agro Lestari Tbk.,Consumer Defensive,8550.0,16456085405696,8.807260,14.032496,6.57,NaN,0.045,0.707,0.05565,0.11581,0.06341,2.956,0.211,0.4718,3.332,1.798,0.20315,IDR,IDR,0.094939,-0.051851,1.471714e+12,4.0,3.428610e+12,0.119653,2.861011,17.184145,0.000000,12.266315,0.000000,-1.256952,568.00,6.643275,6,0.401490,2025-12-31,2026-10-21
1,ABMM,ABM Investama Tbk.,MINING,Coal Mining,Energi,Batu Bara,"Minyak, Gas & Batu Bara",Produksi Batu Bara,2500.0,2480.0,2500.0,1633000.0,912.0,4.052384e+09,0.008065,-0.090909,-0.087591,-0.023438,-0.180328,-0.116608,-0.363868,0.597444,0.054852,-0.038462,-0.100719,4.78674,0.43084,6.882912e+12,0.684975,0.304738,4.89376,0.38074,1.88554,3190.0,2120.0,2026-10-21T00:00:00,0.088040,14.97,ABMM.JK,ABM Investama Tbk.,Energy,2720.0,7488608927744,5.300388,NaN,3.53,103.092,0.104,2.965,0.01878,0.01975,0.07928,43471.070,-0.653,0.1940,1.138,0.987,0.09460,IDR,USD,-0.104469,-0.360419,7.061632e+07,4.0,1.594795e+08,0.153616,4.192448,1.990922,110.792435,134.306572,47.153747,4.334112,97.00,3.566176,5,0.354026,2025-12-31,2026-10-21
2,ACES,Aspirasi Hidup Indonesia Tbk.,"TRADE, SERVICES, & INVESTMENT",Retail Trade,Barang Konsumen Non-Primer,Ritel Khusus,Perdagangan Ritel,Ritel Barang Rumah Tangga,338.0,334.0,338.0,35028400.0,3876.0,1.182309e+10,0.011976,-0.011696,-0.023121,-0.055866,-0.124352,-0.210280,-0.574843,-0.767698,-0.615909,0.011976,-0.175610,7.55283,0.89712,5.786692e+12,0.644295,0.324873,7.41304,0.63747,27.01168,464.0,308.0,2026-10-21T00:00:00,0.121019,39.66,ACES.JK,Aspirasi Hidup Indonesia Tbk.,Consumer Cyclical,360.0,6163340460032,8.064516,6.266318,9.36,15.331,0.025,0.499,0.06339,0.07775,0.08600,4.729,0.110,0.7171,4.945,1.664,0.39990,IDR,IDR,0.085025,0.002195,6.687264e+11,4.0,1.459601e+12,0.168962,2.462634,10.605681,16.408820,29.506030,12.558107,-0.908819,32.01,8.891667,6,0.867122,2025-12-31,2026-10-21
3,ADMF,Adira Dinamika Multi Finance Tbk.,FINANCE,Financial Institution,Keuangan,Pembiayaan Konsumen,Jasa Pembiayaan,Pembiayaan Konsumen,7800.0,7800.0,7800.0,574300.0,200.0,4.476818e+09,0.000000,-0.065868,-0.111111,-0.060241,-0.068657,-0.103448,-0.253589,-0.018868,0.278689,-0.045872,-0.093023,4.66309,0.62303,9.639264e+12,0.429090,0.238454,4.28849,0.74142,2.49923,9150.0,7050.0,2026-10-21T00:00:00,0.145281,5.47,ADMF.JK,Adira Dinamika Multi Finance Tb,Financial Services,7800.0,9562698547200,5.000000,NaN,8.08,126.491,0.278,0.478,0.05296,0.37329,0.23286,NaN,0.222,0.4208,4.965,4.923,0.85522,IDR,IDR,0.147501,-0.012747,1.544938e+12,4.0,2.028150e+11,0.020313,0.293655,NaN,134.623529,156.341133,52.517334,NaN,630.00,8.076923,6,0.635926,2025-12-

## 2) Score the Whole Universe

Factor scores are computed on every fetched stock (`scoring.py`), so a score means "strong against the market". The eligibility screen comes afterwards. Missing inputs are not given a free 50: they pull the factor toward 30 in proportion to what is missing, and inputs that do not apply (leverage for banks) are skipped.


In [3]:
merged_df[merged_df['Ticker'] == 'AKRA.JK']

,Code,Name,SectorName,SubSectorName,NewSectorName,NewIndustryName,NewSubSectorName,NewSubIndustryName,Last,PrevClosingPrice,AdjustedClosingPrice,Volume,Frequency,Value,OneDay,OneWeek,OneMonth,ThreeMonth,SixMonth,OneYear,ThreeYear,FiveYear,TenYear,Mtd,Ytd,Per,Pbr,Capitalization,BetaOneYear,StdevOneYear,PerAnnualized,PsrAnnualized,PcfrAnnualized,AdjustedAnnualHighPrice,AdjustedAnnualLowPrice,LastDate,Roe,FreeFloatPct,Ticker,YF Name,Sector,Current Price,Market Cap,PE Ratio,Forward PE,Dividend Yield,Debt to Equity,Revenue Growth,EPS Growth,ROA,Operating Margin,Net Margin,EV/EBITDA,Beta,Payout Ratio,Current Ratio,Quick Ratio,Free Float Ratio,Currency,Financial Currency,Revenue CAGR 3Y,Net Income CAGR 3Y,Net Income TTM,Net Income Years Positive,FCF,FCF Margin,CFO to Net Income,Interest Coverage,Debt to Equity Calc,Liabilities to Equity,Debt to Assets,Net Debt to EBITDA,Dividend TTM,Dividend Yield Calc,Dividend Years 5Y,Payout Ratio Calc,Statements Latest FY,Fetched On
8,AKRA,AKR Corporindo Tbk.,"TRADE, SERVICES, & INVESTMENT",Wholesale (Durable & Non Durable Goods),Energi,Minyak & Gas,"Minyak, Gas & Batu Bara",Penyimpanan & Distribusi Minyak & Gas,1295.0,1210.0,1295.0,66008600.0,13094.0,8.308607e+10,0.070248,-0.125,-0.109966,-0.109966,-0.113014,0.146018,-0.109966,0.471591,-0.007663,-0.106897,0.027778,9.54532,1.58831,2.599515e+13,0.697152,0.434171,9.08413,0.42141,3.25814,1595.0,1090.0,2026-10-21T00:00:00,0.174845,32.69,AKRA.JK,AKR Corporindo Tbk,Energy,1460.0,28907663261696,10.608152,10.346539,6.76,33.327,0.604,0.259,0.05786,0.04878,0.04912,7.767,-0.397,0.7266,1.326,1.028,0.32161,IDR,IDR,-0.010784,0.009518,2.472616e+12,4.0,2.875560e+12,0.062487,1.593848,45.491719,46.995825,172.362971,15.624797,-0.835957,100.0,6.849315,6,0.799813,2025-12-31,2026-10-21


In [4]:
import scoring

universe = scoring.score_universe(merged_df)
universe[list(scoring.WEIGHTS) + ['Total Score', 'Data Coverage']].describe().round(1)


,Quality Score,Growth Score,Income Score,Valuation Score,Risk Score,Momentum Score,Total Score,Data Coverage
count,356.0,356.0,356.0,356.0,356.0,356.0,356.0,356.0
mean,49.1,47.4,46.0,49.7,50.0,50.1,48.3,0.9
std,19.2,18.5,30.0,21.6,20.1,24.4,14.6,0.1
min,6.8,4.6,13.8,5.5,8.6,0.7,16.5,0.5
25%,35.7,33.3,13.8,31.5,34.3,32.6,35.4,0.9
50%,52.3,47.8,49.2,48.3,49.0,50.4,50.2,1.0
75%,63.7,59.3,74.3,66.6,65.4,66.4,60.9,1.0
max,81.8,97.5,95.2,100.0,91.1,99.7,78.3,1.0


## 3) Apply the Eligibility Screen

The screen runs on the already-scored universe. Edit the thresholds below; a stock with an unknown value for a screened field does not pass.  The debt-to-equity limit applies to every stock, banks included (for financials it uses total liabilities / equity).


In [5]:
SCREEN = {
    'min_market_cap': 1e12,
    'min_roe': 0.10,
    'max_debt_to_equity': 100,  # applies to all, banks included
    'min_frequency': 1000,
    'min_dividend_yield': 5,    # percent
    'min_coverage': 0.6,        # share of scoring inputs that must be present
    'min_pe': None,             # e.g. 5; None = off (loss-makers fail when a PE bound is set)
    'max_pe': None,             # e.g. 20
}

hard_filtered = scoring.apply_screen(universe, SCREEN)
print(f'{len(hard_filtered)} of {len(universe)} stocks pass the screen')


34 of 356 stocks pass the screen


## 4) Factor Scores (0-100)

Six factors, each an average of percentile-ranked inputs:

- **Quality:** ROE, ROA, operating and net margin, cash conversion, earnings consistency
- **Growth:** 3-year revenue and net income CAGR, latest revenue growth
- **Income:** dividend yield, payout sustainability (20-70% is best, above 100% scores 0), 5-year dividend record
- **Valuation** (ranked within sector): PE, PB, EV/EBITDA and FCF yield; loss-makers get no PE credit
- **Risk:** beta, volatility, and for non-financials debt to equity (all stocks), net debt to EBITDA, interest coverage
- **Momentum:** 12-1 month and 6-month return


In [6]:
# Inspect how the screened stocks score on each input
factor_cols = list(scoring.WEIGHTS)
hard_filtered[['Ticker'] + factor_cols + ['Data Coverage', 'Flags']].sort_values('Quality Score', ascending=False).head(10).round(1)


,Ticker,Quality Score,Growth Score,Income Score,Valuation Score,Risk Score,Momentum Score,Data Coverage,Flags
52,BSSR.JK,77.9,22.8,67.2,53.6,90.7,80.8,1.0,free float<10%
263,TAPG.JK,77.7,53.3,72.0,44.6,73.3,72.7,1.0,
214,MARK.JK,77.6,56.0,64.3,42.2,80.4,88.4,1.0,
160,SIDO.JK,75.3,28.7,59.8,72.8,81.6,28.2,1.0,payout>100%
184,TLKM.JK,75.0,31.8,59.5,65.8,58.2,36.2,1.0,payout>100%
68,DMAS.JK,75.0,44.4,52.7,63.7,86.4,85.0,1.0,"payout>100%, negative FCF"
170,SMSM.JK,74.3,45.8,89.4,59.6,85.9,59.3,1.0,
119,LSIP.JK,74.0,66.4,87.3,92.4,86.6,64.4,1.0,
193,ULTJ.JK,73.7,63.6,88.1,42.7,90.4,87.0,1.0,
16,ARNA.JK,73.5,45.8,85.0,65.1,90.6,66.8,1.0,


## 5) Weighted Total Score and Ranking

`Rank` is among the screened stocks; `Universe Rank` is against every fetched stock. `Flags` marks red flags such as loss-making, payout above 100%, negative free cash flow, free float below 10% or thin data.


In [7]:
weights = scoring.WEIGHTS
ranked_stocks = scoring.rank(hard_filtered)
ranked_stocks.head(30)


,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,1,1,LSIP.JK,Consumer Defensive,Makanan & Minuman,Perkebunan & Tanaman Pangan,78.307052,73.977324,92.448743,66.439501,64.355545,86.552104,87.312734,1.000000,,1465.0,9991247364096,4.64380,0.66951,5.665529,0.124455,0.002350,0.063256,0.221236,0.468534,-3.426787,-0.140673,0.048507,-0.180758,0.033088,0.779121,0.413452,2972.0,0.2742,0.403780,5.665529
1,2,3,TOTL.JK,Industrials,Konstruksi Bangunan,Konstruksi Bangunan,73.844025,62.605593,49.266661,83.105216,86.306248,76.492135,90.308989,0.980000,,1440.0,4910399619072,9.43557,3.92598,7.638889,0.440518,0.000000,0.196649,0.653295,0.094114,-3.152304,-0.118750,0.336493,0.155738,0.593220,0.602085,0.436251,1441.0,0.7361,0.302620,7.638889
2,3,4,NCKL.JK,Basic Materials,Barang Baku,Logam & Mineral Mulia,73.090230,71.392043,76.187639,83.712102,33.491795,54.998478,78.558052,1.000000,,800.0,50390292758528,4.70422,0.94929,5.330000,0.220049,24.270630,0.457643,0.242477,0.258554,0.456388,-0.145161,-0.101695,-0.317597,-0.369048,1.213525,0.592013,4316.0,0.0000,0.129130,5.330000
3,4,8,ELSA.JK,Energy,"Minyak, Gas & Batu Bara",Jasa Pengeboran Minyak & Gas,71.801834,55.891563,89.330004,62.913383,72.635696,73.071402,88.810861,1.000000,,670.0,4889994854400,5.63052,0.84747,6.610448,0.160242,12.232719,0.056174,0.238610,0.085660,-1.797610,-0.093525,-0.045455,-0.125000,0.296296,0.771116,0.585342,2990.0,0.3958,0.430910,6.610448
4,5,10,ULTJ.JK,Consumer Defensive,Makanan & Minuman,Produk Susu Olahan,71.754481,73.719890,42.667370,63.634657,87.000602,90.439857,88.108614,1.000000,,2080.0,21628205400064,12.14157,2.72292,6.250000,0.261611,0.529452,0.046226,0.120957,0.143900,-1.002013,-0.004739,0.468531,0.386139,0.567164,0.389508,0.399317,2887.0,0.7520,0.179350,6.250000
5,6,16,BIRD.JK,Industrials,Transportasi,Transportasi Jalanan,69.901524,58.661227,71.276224,63.467706,53.593465,72.060493,93.960674,1.000000,negative FCF,1425.0,3565492371456,5.74199,0.57238,11.649123,0.102920,36.635228,0.166964,0.210629,-0.147769,0.730205,-0.109375,-0.131098,-0.144144,-0.214876,0.569300,0.266760,1563.0,0.6694,0.294400,11.649123
6,7,17,JPFA.JK,Consumer Defensive,Makanan & Minuman,"Ikan, Daging & Produk Unggas",69.460520,59.547256,76.820070,71.792552,43.819191,49.621600,89.747191,1.000000,,1975.0,23117120405504,3.91396,0.96191,7.088608,0.232063,63.556773,0.074281,0.412809,0.036127,1.067318,-0.229075,-0.158654,-0.339623,-0.267782,0.879998,0.520879,6635.0,0.3118,0.574855,7.088608
7,8,19,ASGR.JK,Industrials,Jasa Perindustrian,Jasa Pendukung Bisnis,69.090875,58.371172,73.035714,46.465080,88.589965,84.613764,95.177903,1.000000,,2430.0,3277536624640,10.64346,1.65653,22.139918,0.142291,1.156006,0.009173,0.407414,0.121561,-3.526225,0.138095,0.267905,0.257895,1.223256,0.590397,0.541469,1505.0,1.0734,0.231340,22.139918
8,9,20,ARNA.JK,Industrials,Barang Perindustrian,Produk & Perlengkapan Bangunan,68.983683,73.458229,65.133929,45.794586,66.847649,90.562801,85.008640,1.000000,,505.0,3550138335232,8.89073,2.03891,8.910891,0.242737,3.846758,0.040557,-0.114210,0.124017,-0.074641,-0.009804,0.060924,0.000000,-0.028846,0.317815,0.200000,1012.0,0.7700,0.432901,8.910891
9,10,23,SMSM.JK,Consumer Cyclical,Otomotif & Komponen Otomotif,Suku Cadang Otomotif,68.395333,74.296966,59.637625,45.769295,59.294820,85.932033,89.375896,1.000000,,1680.0,9674574266368,8.14821,2.06043,8.630952,0.243381,4.542194,0.029409,0.098568,0.172039,-0.663605,-0.047059,-0.079545,-0.092437,-0.102493,0.483236,0.249536,1178.0,0.7035,0.414260,8.630952


In [8]:
ranked_market_cap = ranked_stocks[ranked_stocks['Market Cap'].notna()].copy()
ranked_market_cap = ranked_market_cap.sort_values(['Market Cap', 'Total Score'], ascending=[False, False]).reset_index(drop=True)
ranked_market_cap.head(30)

,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,28,130,TLKM.JK,Communication Services,Telekomunikasi,Jasa Telekomunikasi Terintegrasi,56.112411,75.026991,65.776764,31.823187,36.201648,58.225202,59.503745,1.000000,payout>100%,2210.0,217863877033984,12.02680,1.55740,10.098004,0.157555,57.829752,-0.001278,-0.055239,0.259462,0.293136,-0.152000,-0.229091,-0.295681,-0.353659,1.117565,0.498233,41013.0,1.2527,0.471220,10.098004
1,21,78,AADI.JK,Energy,"Minyak, Gas & Batu Bara",Produksi Batu Bara,61.891326,72.248885,64.490882,27.467285,80.592684,75.193168,78.698502,1.000000,,11150.0,86823846019072,6.06214,1.23038,8.981166,0.240840,24.782589,-0.140225,-0.313497,0.108880,-0.330839,-0.043103,0.233333,0.020690,0.383178,0.583753,0.479448,13338.0,0.5818,0.371910,8.981166
2,25,106,HMSP.JK,Consumer Defensive,Rokok,Rokok,58.818133,72.646828,51.794632,39.106912,53.594686,84.848906,59.729391,1.000000,free float<10%,630.0,73280388071424,8.64814,2.88147,8.936508,0.322173,1.209389,0.002871,0.014839,0.059320,-0.551904,0.000000,-0.091549,-0.145695,-0.116438,0.655357,0.413327,2711.0,0.7507,0.075610,8.936508
3,31,142,ADRO.JK,Energy,"Minyak, Gas & Batu Bara",Produksi Batu Bara,54.690709,69.993845,58.877786,16.535344,78.038308,82.929938,60.159176,1.000000,"payout>100%, negative FCF",2440.0,70273206321152,6.91695,0.77041,10.795082,0.120673,18.309622,-0.386216,-0.435823,-0.110723,-0.877068,-0.083969,-0.055118,-0.040000,0.329640,0.660481,0.406523,17222.0,0.7899,0.380620,10.795082
4,12,28,ANTM.JK,Basic Materials,Barang Baku,Logam & Mineral Lainnya,67.319748,62.772088,63.165188,61.138191,46.962507,65.511099,89.840824,1.000000,,2900.0,69689216598016,7.34337,1.69637,7.240988,0.331603,12.909729,0.226016,0.235655,0.049076,-0.486211,-0.160494,-0.111111,-0.325062,-0.173252,1.279177,0.579690,30084.0,0.5669,0.349970,7.240988
5,18,66,UNVR.JK,Consumer Defensive,Produk Rumah Tangga Tidak Tahan Lama,Produk Perawatan Tubuh,63.142575,71.039267,43.907791,40.407485,40.613604,78.157237,94.241573,1.000000,,1585.0,60200253915136,6.18138,17.29023,12.681388,1.964860,14.409475,-0.081466,0.125131,0.154498,-0.218590,-0.154321,-0.198830,-0.238889,-0.331707,0.605541,0.467399,13308.0,2.0510,0.146310,12.681388
6,3,4,NCKL.JK,Basic Materials,Barang Baku,Logam & Mineral Mulia,73.090230,71.392043,76.187639,83.712102,33.491795,54.998478,78.558052,1.000000,,800.0,50390292758528,4.70422,0.94929,5.330000,0.220049,24.270630,0.457643,0.242477,0.258554,0.456388,-0.145161,-0.101695,-0.317597,-0.369048,1.213525,0.592013,4316.0,0.0000,0.129130,5.330000
7,11,25,CPIN.JK,Consumer Defensive,Makanan & Minuman,"Ikan, Daging & Produk Unggas",68.038897,60.400311,69.405599,67.074744,23.079741,70.116740,87.968165,1.000000,,2890.0,47390220353536,6.02948,1.28699,6.228374,0.212443,19.516255,0.075297,0.244471,0.056609,0.073801,-0.118971,-0.124601,-0.391111,-0.439673,0.659570,0.439428,5419.0,0.3965,0.389020,6.228374
8,16,52,TAPG.JK,Consumer Defensive,Makanan & Minuman,Perkebunan & Tanaman Pangan,64.807993,77.685665,44.556739,53.345452,72.728901,73.251126,72.025726,1.000000,,2040.0,40499180208128,9.36599,3.17740,9.852941,0.341950,9.808210,0.068546,0.074887,0.246335,-0.233421,-0.119816,0.186335,-0.027990,0.029650,0.918301,0.552855,4909.0,0.8833,0.410560,9.852941
9,19,68,CMRY.JK,Consumer Defensive,Makanan & Minuman,Produk Susu Olahan,63.003998,71.233518,32.180283,73.168352,58.711986,87.358013,52.025438,1.000000,,4550.0,36102807224320,15.25488,4.70792,6.593407,0.327120,0.006294,0.189094,0.242204,0.115900,-0.475947,-0.047085,-0.080087,-0.082073,-0.150000,0.515827,0.426650,2743.0,0.7179,0.192750,6.593407


In [9]:
ranked_market_up = ranked_stocks[ranked_stocks['Market Cap'].notna()].copy()
ranked_market_up = ranked_market_up[ranked_market_up['ThreeMonth'].fillna(-999) >= 0.1]  # Three month performance greater than or equal to 5%
ranked_market_up = ranked_market_up.sort_values(['OneYear', 'Total Score'], ascending=[True, False]).reset_index(drop=True)
ranked_market_up.head(30)

,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,16,52,TAPG.JK,Consumer Defensive,Makanan & Minuman,Perkebunan & Tanaman Pangan,64.807993,77.685665,44.556739,53.345452,72.728901,73.251126,72.025726,1.00,,2040.0,40499180208128,9.36599,3.17740,9.852941,0.341950,9.808210,0.068546,0.074887,0.246335,-0.233421,-0.119816,0.186335,-0.027990,0.029650,0.918301,0.552855,4909.0,0.8833,0.41056,9.852941
1,23,97,BSSR.JK,Energy,"Minyak, Gas & Batu Bara",Produksi Batu Bara,59.781036,77.910723,53.638718,22.812026,80.846249,90.701316,67.210487,1.00,free float<10%,5200.0,13605799985152,7.83465,2.84700,23.890246,0.473166,0.471870,-0.118944,-0.295209,0.181495,-0.558449,0.051829,0.206294,0.287313,0.323529,0.272884,0.267517,1856.0,1.3460,0.09259,23.890246
2,29,132,POWR.JK,Utilities,Utilitas,Utilitas Listrik,55.747162,65.969012,39.700018,34.808363,82.854422,74.445113,65.052917,1.00,,950.0,15058924994560,12.30583,1.20034,7.761314,0.105282,48.196162,0.001841,-0.002208,0.166045,0.269954,-0.035897,0.287671,0.278912,0.333333,0.505316,0.333320,1824.0,1.0179,0.14678,7.761314
3,21,78,AADI.JK,Energy,"Minyak, Gas & Batu Bara",Produksi Batu Bara,61.891326,72.248885,64.490882,27.467285,80.592684,75.193168,78.698502,1.00,,11150.0,86823846019072,6.06214,1.23038,8.981166,0.240840,24.782589,-0.140225,-0.313497,0.108880,-0.330839,-0.043103,0.233333,0.020690,0.383178,0.583753,0.479448,13338.0,0.5818,0.37191,8.981166
4,13,29,AUTO.JK,Consumer Cyclical,Otomotif & Komponen Otomotif,Suku Cadang Otomotif,67.301554,49.809398,65.769692,56.350842,83.579301,87.301837,89.934457,1.00,,3310.0,15953316282368,6.63900,0.93337,7.417523,0.133924,3.808589,0.023259,0.184569,0.070321,-2.126716,-0.011869,0.251880,0.189286,0.435345,0.541136,0.320398,2120.0,0.4571,0.20000,7.417523
5,20,69,DMAS.JK,Real Estate,Properti & Real Estat,Pengembang & Operator Real Estat,62.845781,75.008754,63.743548,44.395317,84.986325,86.408439,52.715356,1.00,"payout>100%, negative FCF",182.0,8772055990272,5.67635,1.37156,9.065934,0.271974,0.000000,-0.121742,-0.130464,-0.023034,-1.389913,0.100529,0.507246,0.518248,0.540741,0.684596,0.410662,40705.0,0.5099,0.17627,9.065934
6,5,10,ULTJ.JK,Consumer Defensive,Makanan & Minuman,Produk Susu Olahan,71.754481,73.719890,42.667370,63.634657,87.000602,90.439857,88.108614,1.00,,2080.0,21628205400064,12.14157,2.72292,6.250000,0.261611,0.529452,0.046226,0.120957,0.143900,-1.002013,-0.004739,0.468531,0.386139,0.567164,0.389508,0.399317,2887.0,0.7520,0.17935,6.250000
7,2,3,TOTL.JK,Industrials,Konstruksi Bangunan,Konstruksi Bangunan,73.844025,62.605593,49.266661,83.105216,86.306248,76.492135,90.308989,0.98,,1440.0,4910399619072,9.43557,3.92598,7.638889,0.440518,0.000000,0.196649,0.653295,0.094114,-3.152304,-0.118750,0.336493,0.155738,0.593220,0.602085,0.436251,1441.0,0.7361,0.30262,7.638889
8,15,50,MARK.JK,Healthcare,Barang Perindustrian,Mesin & Komponen Perindustrian,65.024999,77.615325,42.157738,55.955138,88.428383,80.364165,64.254433,1.00,,1115.0,4237000179712,12.87339,4.84220,8.071749,0.411100,7.330493,0.005496,0.053063,0.269780,0.051830,0.008850,0.157360,0.425000,0.701493,0.432722,0.413890,2502.0,1.0164,0.19732,8.071749
9,8,19,ASGR.JK,Industrials,Jasa Perindustrian,Jasa Pendukung Bisnis,69.090875,58.371172,73.035714,46.465080,88.589965,84.613764,95.177903,1.00,,2430.0,3277536624640,10.64346,1.65653,22.139918,0.142291,1.156006,0.009173,0.407414,0.121561,-3.526225,0.138095,0.267905,0.257895,1.223256,0.590397,0.541469,1505.0,1.0734,0.23134,22.139918


## 6) Generate Sector-Relative Rankings

This optional section ranks stocks within each sector to reduce concentration effects and make peer comparisons more meaningful.


In [10]:
# count how many ranked stocks are in each sector
ranked_stocks['Sector'].value_counts()

Sector
Consumer Defensive        9
Energy                    6
Basic Materials           5
Consumer Cyclical         5
Industrials               4
Communication Services    2
Healthcare                1
Real Estate               1
Utilities                 1
Name: count, dtype: int64

In [11]:
# count how many ranked stocks are in each sector and sub-industry
ranked_stocks.groupby(['Sector', 'NewSubIndustryName']).size().sort_values(ascending=False)

Sector                  NewSubIndustryName                   
Energy                  Produksi Batu Bara                       4
Consumer Defensive      Perkebunan & Tanaman Pangan              2
                        Produk Susu Olahan                       2
Consumer Cyclical       Suku Cadang Otomotif                     2
Consumer Defensive      Ikan, Daging & Produk Unggas             2
Real Estate             Pengembang & Operator Real Estat         1
Industrials             Transportasi Jalanan                     1
                        Produk & Perlengkapan Bangunan           1
                        Konstruksi Bangunan                      1
                        Jasa Pendukung Bisnis                    1
Healthcare              Mesin & Komponen Perindustrian           1
Energy                  Penyimpanan & Distribusi Minyak & Gas    1
                        Jasa Pengeboran Minyak & Gas             1
Consumer Defensive      Rokok                                    1


In [12]:
for sector, sector_df in ranked_stocks.groupby('Sector', dropna=False):
    print(f"\n=== Sector: {sector} ({len(sector_df)} rows) ===")
    display(sector_df.reset_index(drop=True))



=== Sector: Basic Materials (5 rows) ===


,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,3,4,NCKL.JK,Basic Materials,Barang Baku,Logam & Mineral Mulia,73.090230,71.392043,76.187639,83.712102,33.491795,54.998478,78.558052,1.000000,,800.0,50390292758528,4.70422,0.94929,5.330000,0.220049,24.270630,0.457643,0.242477,0.258554,0.456388,-0.145161,-0.101695,-0.317597,-0.369048,1.213525,0.592013,4316.0,0.0000,0.129130,5.330000
1,12,28,ANTM.JK,Basic Materials,Barang Baku,Logam & Mineral Lainnya,67.319748,62.772088,63.165188,61.138191,46.962507,65.511099,89.840824,1.000000,,2900.0,69689216598016,7.34337,1.69637,7.240988,0.331603,12.909729,0.226016,0.235655,0.049076,-0.486211,-0.160494,-0.111111,-0.325062,-0.173252,1.279177,0.579690,30084.0,0.5669,0.349970,7.240988
2,14,39,AVIA.JK,Basic Materials,Barang Baku,Barang Kimia Khusus,66.015759,71.715989,53.324557,54.738035,47.436670,87.646083,76.335640,1.000000,,354.0,20930411626496,10.74143,2.12685,6.497175,0.195246,0.903915,0.066642,0.076553,0.152100,-1.807946,-0.023669,-0.035088,-0.153846,-0.221698,0.685579,0.356135,2016.0,0.7184,0.151104,6.497175
3,24,99,ESSA.JK,Basic Materials,Barang Baku,Barang Kimia Dasar,59.594485,69.745251,51.527162,29.767161,41.987253,76.407539,86.235955,1.000000,,560.0,9647105769472,9.19731,0.90524,9.285714,0.108891,0.040139,-0.261169,-0.337941,0.325771,-1.418096,-0.133333,-0.147541,-0.380952,-0.154472,0.999149,0.593771,7151.0,0.9545,0.301770,9.285714
4,32,146,PSAB.JK,Basic Materials,Barang Baku,Emas,54.147403,52.284188,37.807650,61.272252,46.345484,49.933924,63.882394,0.879167,free float<10%,484.0,12806640369664,NaN,1.47672,27.892562,0.816945,69.475064,0.422705,NaN,0.293124,0.165311,-0.196296,0.000000,-0.245217,-0.294309,0.902602,0.716437,5099.0,0.8443,0.075000,27.892562



=== Sector: Communication Services (2 rows) ===


,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,28,130,TLKM.JK,Communication Services,Telekomunikasi,Jasa Telekomunikasi Terintegrasi,56.112411,75.026991,65.776764,31.823187,36.201648,58.225202,59.503745,1.0,payout>100%,2210.0,217863877033984,12.02680,1.55740,10.098004,0.157555,57.829752,-0.001278,-0.055239,0.259462,0.293136,-0.152000,-0.229091,-0.295681,-0.353659,1.117565,0.498233,41013.0,1.2527,0.471220,10.098004
1,34,180,SCMA.JK,Communication Services,Media & Hiburan,Penyiaran,49.781372,61.289294,53.041127,32.140908,15.378761,67.436587,54.775281,1.0,payout>100%,171.0,10869563981824,11.61945,1.55003,12.280702,0.145045,0.556686,-0.011561,-0.030600,0.053297,-1.195898,-0.148352,-0.261905,-0.458042,-0.578804,1.475750,0.666974,3792.0,1.3509,0.065648,12.280702



=== Sector: Consumer Cyclical (5 rows) ===


...

,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,2,3,TOTL.JK,Industrials,Konstruksi Bangunan,Konstruksi Bangunan,73.844025,62.605593,49.266661,83.105216,86.306248,76.492135,90.308989,0.98,,1440.0,4910399619072,9.43557,3.92598,7.638889,0.440518,0.000000,0.196649,0.653295,0.094114,-3.152304,-0.118750,0.336493,0.155738,0.593220,0.602085,0.436251,1441.0,0.7361,0.302620,7.638889
1,6,16,BIRD.JK,Industrials,Transportasi,Transportasi Jalanan,69.901524,58.661227,71.276224,63.467706,53.593465,72.060493,93.960674,1.00,negative FCF,1425.0,3565492371456,5.74199,0.57238,11.649123,0.102920,36.635228,0.166964,0.210629,-0.147769,0.730205,-0.109375,-0.131098,-0.144144,-0.214876,0.569300,0.266760,1563.0,0.6694,0.294400,11.649123
2,8,19,ASGR.JK,Industrials,Jasa Perindustrian,Jasa Pendukung Bisnis,69.090875,58.371172,73.035714,46.465080,88.589965,84.613764,95.177903,1.00,,2430.0,3277536624640,10.64346,1.65653,22.139918,0.142291,1.156006,0.009173,0.407414,0.121561,-3.526225,0.138095,0.267905,0.257895,1.223256,0.590397,0.541469,1505.0,1.0734,0.231340,22.139918
3,9,20,ARNA.JK,Industrials,Barang Perindustrian,Produk & Perlengkapan Bangunan,68.983683,73.458229,65.133929,45.794586,66.847649,90.562801,85.008640,1.00,,505.0,3550138335232,8.89073,2.03891,8.910891,0.242737,3.846758,0.040557,-0.114210,0.124017,-0.074641,-0.009804,0.060924,0.000000,-0.028846,0.317815,0.200000,1012.0,0.7700,0.432901,8.910891



=== Sector: Real Estate (1 rows) ===


,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,20,69,DMAS.JK,Real Estate,Properti & Real Estat,Pengembang & Operator Real Estat,62.845781,75.008754,63.743548,44.395317,84.986325,86.408439,52.715356,1.0,"payout>100%, negative FCF",182.0,8772055990272,5.67635,1.37156,9.065934,0.271974,0.0,-0.121742,-0.130464,-0.023034,-1.389913,0.100529,0.507246,0.518248,0.540741,0.684596,0.410662,40705.0,0.5099,0.17627,9.065934



=== Sector: Utilities (1 rows) ===


,Rank,Universe Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Data Coverage,Flags,Current Price,Market Cap,Per,Pbr,Dividend Yield Best,Roe,Leverage,Revenue CAGR 3Y,Net Income CAGR 3Y,FCF Margin,Net Debt to EBITDA,OneMonth,ThreeMonth,SixMonth,OneYear,BetaOneYear,StdevOneYear,Frequency,Payout Ratio,Free Float Ratio,Dividend Yield
0,29,132,POWR.JK,Utilities,Utilitas,Utilitas Listrik,55.747162,65.969012,39.700018,34.808363,82.854422,74.445113,65.052917,1.0,,950.0,15058924994560,12.30583,1.20034,7.761314,0.105282,48.196162,0.001841,-0.002208,0.166045,0.269954,-0.035897,0.287671,0.278912,0.333333,0.505316,0.33332,1824.0,1.0179,0.14678,7.761314


In [13]:
# Portfolio selection: top N of the ranking by default, or set manual_tickers to override.
top_n = 10
manual_tickers = []  # e.g. ['TAPG.JK', 'SMSM.JK']; leave empty to use the top N

total_investment = 100_000_000  # IDR
lot_size = 100                  # IDX trades in lots of 100 shares

# Allocation settings
score_weight_strength = 0.70  # 0.30 conservative, 0.50 balanced, 0.70 aggressive
min_position_pct = 10         # Minimum target allocation per stock
max_position_pct = 35         # Maximum target allocation per stock

# Allocation score weights
total_score_weight = 0.50
market_cap_weight = 0.20
liquidity_weight = 0.30

if manual_tickers:
    missing = sorted(set(manual_tickers) - set(ranked_stocks['Ticker']))
    if missing:
        print(f"Not in ranked_stocks (dropped by filters or score cutoff), skipped: {missing}")
    portfolio_stocks = ranked_stocks[ranked_stocks['Ticker'].isin(manual_tickers)].copy()
else:
    portfolio_stocks = ranked_stocks.sort_values('Total Score', ascending=False).head(top_n).copy()

num_stocks = len(portfolio_stocks)
if num_stocks == 0:
    print("No stocks to allocate.")
else:
    # Widen the bounds when few stocks are selected so they stay feasible.
    min_weight = min(min_position_pct / 100, 1 / num_stocks)
    max_weight = max(max_position_pct / 100, 1 / num_stocks)
    if (min_weight, max_weight) != (min_position_pct / 100, max_position_pct / 100):
        print(f"Bounds adjusted for {num_stocks} stocks: min {min_weight:.1%}, max {max_weight:.1%}")

    total_score_component = portfolio_stocks['Total Score'].fillna(0).clip(lower=0)
    market_cap_component = portfolio_stocks['Market Cap'].fillna(0).clip(lower=0).rank(pct=True) * 100
    liquidity_component = portfolio_stocks['Frequency'].fillna(0).clip(lower=0).rank(pct=True) * 100
    portfolio_stocks['Allocation Score'] = (
        total_score_weight * total_score_component
        + market_cap_weight * market_cap_component
        + liquidity_weight * liquidity_component
    )

    allocation_scores = portfolio_stocks['Allocation Score'].fillna(0).clip(lower=0)
    equal_weights = pd.Series(1 / num_stocks, index=portfolio_stocks.index)
    if allocation_scores.sum() > 0:
        allocation_weights = allocation_scores / allocation_scores.sum()
        blended_weights = ((1 - score_weight_strength) * equal_weights) + (score_weight_strength * allocation_weights)
    else:
        blended_weights = equal_weights

    # Clamp to [min, max], redistributing the remainder over unclamped names until stable.
    final_weights = pd.Series(index=portfolio_stocks.index, dtype=float)
    remaining_weights = blended_weights.copy()
    remaining_total = 1.0
    while len(remaining_weights) > 0:
        proposed_weights = remaining_total * remaining_weights / remaining_weights.sum()
        below_min = proposed_weights < min_weight - 1e-12
        above_max = proposed_weights > max_weight + 1e-12
        if not below_min.any() and not above_max.any():
            final_weights.loc[proposed_weights.index] = proposed_weights
            break

        # Clamp only the worse violation side per pass, so one pass cannot over-fill.
        if above_max.sum() >= below_min.sum():
            clamp, bound = above_max, max_weight
        else:
            clamp, bound = below_min, min_weight
        final_weights.loc[clamp[clamp].index] = bound
        remaining_total = 1 - final_weights.dropna().sum()
        remaining_weights = remaining_weights.drop(clamp[clamp].index)

    portfolio_stocks['Portfolio Percentage'] = final_weights * 100
    portfolio_stocks['Allocation'] = total_investment * final_weights
    # Whole lots only; leftover cash is reported below.
    lots = (portfolio_stocks['Allocation'] / (portfolio_stocks['Current Price'] * lot_size)).apply(np.floor)
    portfolio_stocks['Lots'] = lots.astype(int)
    portfolio_stocks['Shares to Buy'] = portfolio_stocks['Lots'] * lot_size
    portfolio_stocks['Value'] = portfolio_stocks['Shares to Buy'] * portfolio_stocks['Current Price']

    print(f"Total Investment: {total_investment:,.0f} | Invested: {portfolio_stocks['Value'].sum():,.0f} | "
          f"Cash left: {total_investment - portfolio_stocks['Value'].sum():,.0f}")
    display(portfolio_stocks[['Ticker', 'Sector', 'Total Score', 'Allocation Score', 'Market Cap', 'Frequency',
                              'Current Price', 'Portfolio Percentage', 'Allocation', 'Lots', 'Shares to Buy', 'Value']]
            .sort_values('Portfolio Percentage', ascending=False).reset_index(drop=True))


Total Investment: 100,000,000 | Invested: 99,509,000 | Cash left: 491,000


,Ticker,Sector,Total Score,Allocation Score,Market Cap,Frequency,Current Price,Portfolio Percentage,Allocation,Lots,Shares to Buy,Value
0,LSIP.JK,Consumer Defensive,78.307052,74.153526,9991247364096,2972.0,1465.0,10.0,10000000.0,68,6800,9962000.0
1,TOTL.JK,Industrials,73.844025,55.922012,4910399619072,1441.0,1440.0,10.0,10000000.0,69,6900,9936000.0
2,ELSA.JK,Energy,71.801834,67.900917,4889994854400,2990.0,670.0,10.0,10000000.0,149,14900,9983000.0
3,ULTJ.JK,Consumer Defensive,71.754481,69.877241,21628205400064,2887.0,2080.0,10.0,10000000.0,48,4800,9984000.0
4,BIRD.JK,Industrials,69.901524,55.950762,3565492371456,1563.0,1425.0,10.0,10000000.0,70,7000,9975000.0
5,JPFA.JK,Consumer Defensive,69.460520,82.730260,23117120405504,6635.0,1975.0,10.0,10000000.0,50,5000,9875000.0
6,ASGR.JK,Industrials,69.090875,48.545438,3277536624640,1505.0,2430.0,10.0,10000000.0,41,4100,9963000.0
7,ARNA.JK,Industrials,68.983683,41.491842,3550138335232,1012.0,505.0,10.0,10000000.0,198,19800,9999000.0
8,SMSM.JK,Consumer Cyclical,68.395333,52.197666,9674574266368,1178.0,1680.0,10.0,10000000.0,59,5900,9912000.0
9,NCKL.JK,Basic Materials,73.090230,83.545115,50390292758528,4316.0,800.0,10.0,10000000.0,124,12400,9920000.0


## 7) Visualize the Results

Charts live in `viz.py`. The last cell also saves them as one standalone page at `reports/idx_report.html`.


In [14]:
import importlib, viz
importlib.reload(viz)  # pick up edits to viz.py without restarting the kernel

viz.score_breakdown(ranked_stocks, weights, n=15).show()


In [15]:
viz.factor_heatmap(ranked_stocks, n=15).show()


In [16]:
viz.yield_vs_roe(ranked_stocks, portfolio_stocks).show()


In [17]:
viz.sector_mix(ranked_stocks).show()


In [18]:
viz.allocation(portfolio_stocks).show()


In [19]:
figures = [
    viz.score_breakdown(ranked_stocks, weights, n=15),
    viz.factor_heatmap(ranked_stocks, n=15),
    viz.yield_vs_roe(ranked_stocks, portfolio_stocks),
    viz.sector_mix(ranked_stocks),
    viz.allocation(portfolio_stocks),
]
print('Saved', viz.save_report(figures))


Saved reports/idx_report.html
